# Load Sample Source

Illustrate how to load a netCDF file for one source using `xarray` or `rasterio`, and how to grab the last frame to make an "instant" rupture version.

Also write it out as a GeoClaw dtopo file with ASCII raster format, as described in the [GeoClaw Documentation](https://www.clawpack.org/dtopo.html), and make some plots using GeoClaw tools.

Adapt this code to write it out to whatever format you need.

In [ ]:
%matplotlib inline

In [ ]:
from pylab import *
from pathlib import Path

In [ ]:
dtopo_dir = './dtopofiles_nc'  # path to unzipped directory

In [ ]:
event = 'BL13M'
path = Path(dtopo_dir) / f'{event}.nc'
print(path)

## Open with xarray:

Using [xarray](https://docs.xarray.dev/en/stable/).

Also illustrate how to grab the final static deformation.

In [ ]:
import xarray as xr
with xr.open_dataset(path, decode_timedelta=False) as dtopo_xr:
    print(dtopo_xr)

    # To grab only the final deformation for use as an "instant" event:
    
    print(f"\nThe dz array has shape {dtopo_xr.variables['dz'].shape}")
    dz_static = dtopo_xr.variables['dz'][-1,:,:]
    print(f"The dz_static array contains the final (static) deformation, with shape {dz_static.shape}")
    print(f"The maximum static uplift is {dz_static.max():.2f} meters")

## Open with rasterio:

Or use [rasterio](https://rasterio.readthedocs.io/en/stable/), another standard Python package that supports reading netCDF easily.

In [ ]:
import rasterio
with rasterio.open(path) as src:
    print(f"Coordinate Reference System (CRS): {src.crs}")
    print(f"Bounds: {src.bounds}")
    print(f"Number of bands: {src.count}")
    print(f"Width/Height: {src.width}x{src.height}")
    
    meta = src.meta
    print('\nsrc.meta = \n')
    for k in meta.keys():
        print(f'{k:<30}{meta[k]}')
    
    tags = src.tags()
    print('\nsrc.tags = \n')
    for k in tags.keys():
        print(f'{k:<30}:  {tags[k]}')

## Load with GeoClaw

See [GeoClaw Documentation](https://www.clawpack.org/dtopo.html) for info on the file formats and other tools available.

In [ ]:
from clawpack.geoclaw import dtopotools

dtopo = dtopotools.DTopography(path, dtopo_type=4)  # 4 ==> netcdf format

### Rewrite as GeoClaw ascii file

In [ ]:
fname = f'{event}.dtt3'
dtopo.write(fname, dtopo_type=3)

In [ ]:
sizeMB = Path(fname).stat().st_size / 1e6
print(f'{fname} has size {sizeMB:.1f} MB')

## Capture only the final deformation for an "instant" rupture

In [ ]:
dz_static = dtopo.dZ
print(f"\nThe dtopo.dZ array has shape {dtopo.dZ.shape}")
dz_static = dz_static = dtopo.dZ[-1,:,:]
print(f"The dz_static array contains the final (static) deformation, with shape {dz_static.shape}")
print(f"The maximum static uplift is {dz_static.max():.2f} meters")

# Create a new dtopo file for an instant rupture at time 1 second:

import copy
dtopo_instant = copy.copy(dtopo)  # so we have the same X and Y arrays
dtopo_instant.dZ = dtopo_instant.dZ[-2:-1, :,:]  # only the last dz
dtopo_instant.times = [1.0]  # desired time of instant rupture
print(f"\nNew dtopo_instant.dZ has shape {dtopo_instant.dZ.shape}" \
        + f" with maximum {dtopo_instant.dZ.max():.1f}")
print(f"Instant displacement with specified times {dtopo_instant.times}")

fname = f'{event}_instant.dtt3'
dtopo_instant.write(fname, dtopo_type=3)
sizeMB = Path(fname).stat().st_size / 1e6
print(f'{fname} has size {sizeMB:.1f} MB')

## Plots of deformation

Using the GeoClaw `dtopotools` module tools.

In [ ]:
coast = load('../topo/CSZ_coast.npy')  # precomputed coastline

In [ ]:
# time to plot deformation
#tplot = dtopo.times.max()  # for final static deformation
tplot1 = dtopo.times.max() / 3.
tplot2 = dtopo.times.max()

fig,axs = subplots(1,2,figsize=(10,6))
for ax in axs:
    ax.plot(coast[:,0], coast[:,1], 'g', linewidth=0.9)
    ax.set_aspect(1/cos(45*pi/180))
    ax.set_xlim(-130,-121)
    ax.set_ylim(39,50)

dtopo.plot_dZ_colors(t=tplot1, axes=axs[0], dZ_interval=100, cmax_dZ=10);
axs[0].set_title(f'{event}\nSeafloor deformation dz\nat time t = {tplot1:.1f} seconds');

dtopo.plot_dZ_colors(t=tplot2, axes=axs[1], dZ_interval=100, cmax_dZ=10);
axs[1].set_title(f'{event}\nSeafloor deformation dz\nat time tfinal = {tplot2:.1f} seconds');

**To Do:** Add an animation.

## Vertical deformation at Lagoon Creek

In [ ]:
# make interpolating function of (x,y,t):
dtopo_fcn = dtopo.make_function()

# Evaluate at desired location (gauge 1 location, on beach):
xg = -124.102
yg = 41.596
tg = arange(0, dtopo.times.max(), 1)  # for time series covering deformation time, with dt=1 sec
dz = dtopo_fcn(xg, yg, tg)

figure(figsize=(8,3))
plot(tg, dz, 'g')
grid(True)
xlabel('time (seconds)')
ylabel(f'vertical deformation dz (meters)')
title(f'vertical deformation dz at xg={xg:.5f}, yg={yg:.5f}');